In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

In [ ]:
path = 'Tweets.csv'
parent_df=pd.read_csv(path)
parent_df.head(),parent_df.shape

In [ ]:
df=parent_df[['airline_sentiment','text']]
df.head()

In [ ]:
df['airline_sentiment'].unique()

In [ ]:
df.shape

TEXT CLEANING

In [ ]:
import re
import html

def clean_text_for_audit(text):
    if not isinstance(text, str):
        return ""

    # 1. Decode HTML entities (e.g., &amp; -> &)
    text = html.unescape(text)

    # 2. Remove URLs and Email addresses
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)

    # 3. Remove weird whitespace but keep newlines as spaces
    text = text.replace('\n', ' ').replace('\r', ' ').replace('\t', ' ')

    # 4. Remove non-printable characters
    text = "".join(char for char in text if char.isprintable())

    # 5. Normalize multiple spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Apply to your DataFrame
df['clean_text'] = df['text'].apply(clean_text_for_audit)

# Deduplication (Crucial: same text with different labels is an automatic error)
df = df.drop_duplicates(subset=['clean_text', 'airline_sentiment'])##removes rows where both the cleaned text AND sentiment label are identical.
print(f"Data cleaned. Remaining samples: {len(df)}")

In [ ]:
df.shape

In [ ]:
df.head()

Text → Embeddings (Semantic Meaning)

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-mpnet-base-v2")
embeddings = model.encode(
    df["clean_text"].tolist(),
    batch_size=32,
    convert_to_numpy=True,
    show_progress_bar=True
)
df["embedding"] = list(embeddings)


In [ ]:
df.head()

OPTUNA FOR UMAP AND HDBSACN

In [ ]:
!pip install optuna

In [ ]:
import optuna

In [ ]:
!pip install umap-learn hdbscan scikit-learn


In [ ]:
import umap,hdbscan
from sklearn.metrics import silhouette_score

In [ ]:
  embeddings = np.stack(df['embedding'].values)  # Stack 1D → 2D
  print(embeddings.shape)  # Now (14419, 768)


In [ ]:
def objective(trial):
    # 1. UMAP: Focus on density (min_dist=0) and enough dimensions for nuance
    n_neighbors = trial.suggest_int('n_neighbors', 10, 50)# Local structure balance
    n_components = trial.suggest_int('n_components', 5, 30) # output dimensions range

    reducer = umap.UMAP(
        n_neighbors=n_neighbors,
        min_dist=0.0, # Keeps clusters tight for auditing
        n_components=n_components,
        metric='cosine',
        random_state=42,
        n_jobs=1
    )
    emb_reduced = reducer.fit_transform(embeddings)

    # 2. HDBSCAN: Let min_samples be smaller for more sensitivity
    min_cluster_size = trial.suggest_int('min_cluster_size', 10, 40)

    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=2, # Sensitivity to small groups
        metric='euclidean',
        prediction_data=True
    )
    labels = clusterer.fit_predict(emb_reduced)

    # 3. Enhanced Scoring logic
    # Filter out noise for the silhouette calculation
    mask = labels >= 0
    n_clusters = len(np.unique(labels[mask]))
    outlier_pct = np.sum(labels == -1) / len(labels)

    if n_clusters < 2 or outlier_pct > 0.3: # We don't want >30% noise
        return -2.0 # Heavy penalty

    # Calculate silhouette ONLY on clustered points
    sil = silhouette_score(emb_reduced[mask], labels[mask])

    # Objective: High silhouette, Low outliers, reasonable cluster count
    # We want clusters, but not 100 clusters (which would be over-fitting)
    score = sil - (outlier_pct * 2) + (0.05 * min(n_clusters, 20))

    return score

In [ ]:
# Create and run the study
study = optuna.create_study(direction="maximize",sampler=optuna.samplers.TPESampler(seed=42),study_name="Label_Audit_Optimization")
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f"Best Score: {study.best_value}")
print(f"Best Hyperparameters: {study.best_params}")

In [ ]:
print(f"Best Score: {study.best_value:.4f}")
print(f"Best Hyperparameters: {study.best_params}")
print(f"Best Trial Number: {study.best_trial.number}")
print(f"All trials: {len(study.trials)}")


In [ ]:
import optuna.visualization as vis

# Small individual plots
fig1 = vis.plot_optimization_history(study)
fig1.update_layout(width=600, height=400, font_size=9)
fig1.show()

fig2 = vis.plot_param_importances(study)
fig2.update_layout(width=600, height=400, font_size=9)
fig2.show()

fig3 = vis.plot_slice(study)
fig3.update_layout(width=600, height=400, font_size=9)
fig3.show()

fig4 = vis.plot_contour(study)
fig4.update_layout(width=600, height=400, font_size=9)
fig4.show()


In [ ]:
# SINGLE CELL - Saves everything forever
import json
from google.colab import files

# Save params
results = {"best_score": study.best_value, "best_params": study.best_params}
with open("label_audit_complete.json", "w") as f:
    json.dump(results, f, indent=2)

# Save plots
vis.plot_optimization_history(study).write_html("1_history.html")
vis.plot_param_importances(study).write_html("2_importance.html")
vis.plot_slice(study).write_html("3_slice.html")
vis.plot_contour(study).write_html("4_contour.html")

# Download ZIP
!zip results.zip label_audit_complete.json 1_*.html 2_*.html 3_*.html 4_*.html
files.download("results.zip")


In [ ]:
# Save FULL dataset with all columns (clusters, minorities, flags)
df.to_csv('/content/drive/MyDrive/airline_audit_complete.csv', index=False)

FINAL UMAP AND CLUSTERING

In [ ]:
bp = study.best_params
print(bp)

In [ ]:
final_reducer = umap.UMAP(n_neighbors=bp['n_neighbors'], n_components=bp['n_components'], min_dist=0.0, metric='cosine')
final_emb = final_reducer.fit_transform(embeddings)

In [ ]:
final_emb.shape

In [ ]:
final_clusterer = hdbscan.HDBSCAN(min_cluster_size=bp['min_cluster_size'], min_samples=2)
df['cluster'] = final_clusterer.fit_predict(final_emb)


In [ ]:
"""
37 stable clusters (0-36)
Only 1 outlier (-1) in sample = 2.6% outlier rate
min_cluster_size=40 working perfectly
"""
df['cluster'].unique()

In [ ]:
df.head()

In [ ]:
# 1. For EACH cluster, find the MOST COMMON label (the "mode")
cluster_modes = df[df.cluster != -1].groupby('cluster')['airline_sentiment'].agg(lambda x: x.mode()[0]).to_dict()
# cluster_modes = {0: 'positive', 2: 'negative', 5: 'positive', ...}


In [ ]:
print(cluster_modes)

In [ ]:
# 2. Flag samples whose label DISAGREES with their cluster's majority
df['is_minority'] = df.apply(lambda x: x.cluster != -1 and x.airline_sentiment != cluster_modes.get(x.cluster), axis=1)


In [ ]:
df['is_minority'].value_counts()


In [ ]:
non_outlier=df[df['cluster']!=-1]
outlier=df[df['cluster']==-1]

In [ ]:
print(f"non_oulier records:{len(non_outlier)},outlier records : {len(outlier)}")
print(f"percentage of outliers : {len(outlier)/len(df)*100}")

In [ ]:
## ONLY FOR VISUALISATION OF CLUSTERS
# Reduce to 2D for visualization (keep your tuned params!)
vis_umap = umap.UMAP(
    n_neighbors=15,
    n_components=2,  # 2D for plot
    min_dist=0.1,
    metric='cosine',
    random_state=42
)
embed_2d = vis_umap.fit_transform(embeddings)

# Plot with cluster colors + minority markers
plt.figure(figsize=(8, 5))
scatter = plt.scatter(embed_2d[:, 0], embed_2d[:, 1],
                     c=df['cluster'], cmap='tab20', s=20, alpha=0.7,
                     )

# Highlight minorities (red edges) + outliers (black)
minority_mask = df['is_minority']
outlier_mask = df['cluster'] == -1
plt.scatter(embed_2d[minority_mask, 0], embed_2d[minority_mask, 1],
           s=100, facecolors='none', edgecolors='red', linewidth=2, label='Minority')
plt.scatter(embed_2d[outlier_mask, 0], embed_2d[outlier_mask, 1],
           s=150, facecolors='black', label='Outliers')

plt.title('Label Auditing: ', fontsize=16)
plt.legend()
plt.tight_layout()
# plt.savefig('/content/drive/MyDrive/Label_Auditing_Project/cluster_visualization.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Show ONLY clusters (no outliers, no minority highlighting)

# Remove outliers

cluster_mask = df['cluster'] != -1

# Filter data

embed_clusters = embed_2d[cluster_mask]
cluster_labels = df.loc[cluster_mask, 'cluster']

# Plot clusters

plt.figure(figsize=(8, 5))
plt.scatter(embed_clusters[:, 0], embed_clusters[:, 1],
c=cluster_labels,
cmap='tab20',
s=20,
alpha=0.7)

plt.title('UMAP Visualization of Clusters (Outliers Removed)', fontsize=14)
plt.colorbar(label='Cluster ID')
plt.tight_layout()
plt.show()


In [ ]:
from sentence_transformers import CrossEncoder

In [ ]:
df.columns

In [ ]:

# --- CROSS-ENCODER VALIDATION ---
ce_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
label_desc = {
    "positive": "Positive sentiment, praise, or satisfaction.",
    "negative": "Negative sentiment, complaint, or disappointment.",
    "neutral": "Neutral, factual, or balanced statement."
}

def get_ce_margin(row):
    # Only run on suspicious samples to save time
     # Skip clean samples (saves 75% compute time!)
    if not row['is_minority'] and row['cluster'] != -1:
      return pd.Series({
        'ce_predicted_label': row['airline_sentiment'],
        'ce_margin': 1.0,
    }) # High confidence = "safe"

    # Score text against ALL 3 label descriptions
    pairs = [[row['clean_text'], desc] for desc in label_desc.values()]

    scores = ce_model.predict(pairs)## [0.05, 0.92, 0.23] = terrible text loves "negative"
    score_dict = dict(zip(label_desc.keys(), scores))# {'positive':0.05, 'negative':0.92, 'neutral':0.23}

    # Predicted label (highest score)
    predicted_label = max(score_dict, key=score_dict.get)

    assigned = score_dict[row['airline_sentiment']]# Given label score
    best_alt = max([v for k,v in score_dict.items() if k != row['airline_sentiment']])# Best other label
    margin = assigned - best_alt
    return pd.Series({
        'ce_predicted_label': predicted_label,
        'ce_margin': margin,
    })

df[['ce_predicted_label','ce_margin']] = df.apply(get_ce_margin, axis=1)

In [ ]:
df.head()

In [ ]:
# df.drop(columns=['audit_status'],axis=1,inplace=True)

In [ ]:
def audit_verdict(row):
    ##
    if row['ce_margin'] < -0.5 and row['is_minority'] and row['ce_predicted_label'] != row['airline_sentiment']:
        return "HIGH RISK (Mislabeled)"
    elif row['ce_margin'] < -0.5:
        return "MEDIUM RISK (Semantic Conflict)"
    elif row['is_minority']:
        return "MEDIUM RISK (Neighborhood Conflict)"
    elif row['cluster'] == -1:
        return "OUTLIER"
    return "SAFE"
df['audit_status'] = df.apply(audit_verdict, axis=1)

In [ ]:
df.head()

In [ ]:
df.to_csv('/content/drive/MyDrive/final_adiited__complete_new.csv', index=False)

In [ ]:
print(df['audit_status'].value_counts())


In [ ]:
high_risk = df[df['audit_status'] == 'HIGH RISK (Mislabeled)']

high_risk.to_csv('high_risk_airline_audit_complete_final.csv', index=False)